In [1]:
!pip -q install transformers datasets peft accelerate

In [2]:
import pandas as pd
import torch
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from torch.nn.functional import softmax

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["label"] = train["answer"].map(label_map)

print("Q1 =", train.loc[150,"label"])

Q1 = 2


In [6]:
MODEL = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
text = (
    str(train.loc[0,"prompt"])
    + " [SEP] "
    + str(train.loc[0,"B"])
)

print(text)

print("Q2 =", len(text))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Q2 = 407


In [8]:
choices = []

for option in ["A","B","C","D","E"]:
    choices.append(
        str(train.loc[0,"prompt"])
        + " [SEP] "
        + str(train.loc[0,option])
    )

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = enc["input_ids"].unsqueeze(0)
attention_mask = enc["attention_mask"].unsqueeze(0)

print(input_ids.shape)

print("Q3 =", input_ids.shape[1])

torch.Size([1, 5, 128])
Q3 = 5


In [9]:
all_ids = []

for i in range(16):

    choices = []

    for option in ["A","B","C","D","E"]:
        choices.append(
            str(train.loc[i,"prompt"])
            + " [SEP] "
            + str(train.loc[i,option])
        )

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_ids.append(enc["input_ids"])

all_ids = torch.stack(all_ids)

print(all_ids.shape)

print("Q4 =", all_ids.numel())

torch.Size([16, 5, 128])
Q4 = 10240


In [10]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print(outputs.logits.shape)

print("Q5 =", outputs.logits.shape[1])

torch.Size([1, 5])
Q5 = 5


In [12]:
label = torch.tensor([train.loc[0,"label"]])

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=label
)

print(outputs.loss)

print(outputs.loss.shape)

print("Q6 =", outputs.loss.dim())

tensor(1.6181, grad_fn=<NllLossBackward0>)
torch.Size([])
Q6 = 0


In [15]:
!pip install -q -U torchao
!pip install -q -U peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.3 MB/s eta 0:00:00a 0:00:01


In [16]:
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(model, config)

trainable = sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

print("Q7 =", trainable)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


Q7 = 295681


In [17]:
small = train.iloc[:100].copy()


def preprocess(example):

    texts = []

    for option in ["A","B","C","D","E"]:
        texts.append(
            str(example["prompt"])
            + " [SEP] "
            + str(example[option])
        )

    enc = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    enc["labels"] = example["label"]

    return enc


dataset = Dataset.from_pandas(small)

dataset = dataset.map(preprocess)

print(np.array(dataset[0]["input_ids"]).shape)

print("Q8 =", np.array(dataset[0]["input_ids"]).shape[0])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

(5, 128)
Q8 = 5


In [18]:
train32 = train.iloc[:32].copy()

dataset = Dataset.from_pandas(train32)

dataset = dataset.map(preprocess)

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

model = AutoModelForMultipleChoice.from_pretrained(MODEL)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

training_args = TrainingArguments(
    output_dir="mcq",
    max_steps=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    logging_steps=1,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

train_result = trainer.train()

print("Q9 =", trainer.state.global_step)

choices = []

for option in ["A","B","C","D","E"]:
    choices.append(
        str(train.loc[0,"prompt"])
        + " [SEP] "
        + str(train.loc[0,option])
    )

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

with torch.no_grad():

    logits = model(
        input_ids=enc["input_ids"].unsqueeze(0),
        attention_mask=enc["attention_mask"].unsqueeze(0)
    ).logits

prob = softmax(logits, dim=1)

print(prob)

print("Q10 =", round(prob[0,4].item(),4))

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.622394
2,1.723286
3,1.634985
4,1.716283


Q9 = 4
tensor([[0.2385, 0.1811, 0.2242, 0.1879, 0.1684]])
Q10 = 0.1684
